# 01. Dashboard bring-up — `arcui`

`arcui` is the **observability dashboard** for an Arc agent fleet. It is a Starlette app that reads from a durable arcstore mirror (the `Observe` plane) on demand and renders a dashboard for monitoring LLM calls, tool invocations, costs, and audit events. Interactive chat with a live agent is bidirectional over `/ws/chat/{agent_id}`; a read-only team-flow stream is available over `/ws/team`.

This notebook walks the *bring-up* surface — the part you touch when you bolt the dashboard onto an existing process. It does **not** cover the `/ws/chat` protocol or the individual routes in depth. Those are the subject of `02-live-telemetry-attach.ipynb`.

**You will learn:**

- What `arcui` is and where it sits in the Arc dependency graph
- How to build a Starlette app with `create_app()`
- The two-token auth model (operator / viewer) and why it exists
- How to run the server with `serve()` — and how to drive it in-process via `TestClient`
- The high-level routing structure and which compliance controls each surface satisfies
- Operational notes for binding, TLS, and reverse proxies in real deployments

Every cell runs **without an API key** and without binding a real port. Where binding a port is the right thing to demonstrate, we use a `subprocess` with a hard timeout so the notebook always terminates.

**A note on architecture (SPEC-026 FR-5).** `arcui` used to run a live push pipeline — agents streamed telemetry over a WebSocket into an `EventBuffer`/`RollingAggregator`/`ConnectionManager`/`SubscriptionManager` stack, and a third "agent" auth token gated that push channel. That entire pipeline is gone. `arcui` is now a **read-only consumer** of the durable operational record: reads come from `app.state.observe` (an arcstore mirror, backfilled from the durable spool + WORM store at startup and tailed thereafter). The only WebSocket traffic left is `/ws/chat/{agent_id}` (interactive chat, bidirectional) and `/ws/team` (read-only team-flow stream).

## 1. Setup

The standard Arc walkthrough boilerplate. It locates the repo root and adds every `packages/<pkg>/src` (and `tests/`) directory to `sys.path` so the notebook can import `arcui`, `arcllm`, and `arctrust` without a `pip install -e .` cycle.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Confirm the package imports cleanly. If this fails, the boilerplate above did not find the Arc repo root.

In [2]:
import arcui
from arcui import create_app, serve, attach_llm

print(f"arcui version: {arcui.__version__}")
print(f"public surface: {arcui.__all__}")

arcui version: 0.2.0
public surface: ['__version__', 'attach_llm', 'create_app', 'serve']


`arcui.__all__` is the entire stable public API of the package. Three names — `create_app`, `serve`, `attach_llm` — and that is the whole bringup story. We will spend the next sections unpacking each one.

## 2. What `arcui` is

Three layers, top to bottom:

1. **Browser dashboard** — a single-page app served at `/` that renders traces, fleet roster, cost stats, and (via `/ws/chat/{agent_id}`) an interactive chat surface with a running agent.
2. **REST + WebSocket server** — a Starlette app with `GET /api/...` routes for traces, stats, agents, config, export, plus the two WebSocket routes (`/ws/chat/{agent_id}`, `/ws/team`).
3. **The `Observe` plane** — an on-demand arcstore mirror (`app.state.observe`) that backfills from the durable spool + WORM store at lifespan startup and answers trace/stats/agent queries. There is no in-memory event buffer or rolling aggregator to keep warm — "the database is the aggregate."

Where it sits in the Arc graph:

In [3]:
# arcui depends on arcgateway (team roster, embedded gateway runtime) and
# arcstore (the durable Observe mirror). It does not import arcllm or
# arctrust directly — there is no JSONLTraceStore or UIBridgeSink wiring;
# telemetry is written to the arcstore spool elsewhere and arcui only reads
# it back through the Observe plane.
import importlib

deps = {}
for name in ("arcui", "arcgateway", "arcstore"):
    try:
        mod = importlib.import_module(name)
        deps[name] = getattr(mod, "__version__", "<no version>")
    except ImportError as exc:
        deps[name] = f"<missing: {exc}>"

for k, v in deps.items():
    print(f"  {k:12s} {v}")

  arcui        0.2.0
  arcgateway   0.2.0
  arcstore     <no version>


Notice that arcui depends on `arcgateway` for fleet plumbing (team roster, file-event bus, fs watcher). Bringing up the dashboard alongside a real agent fleet means optionally passing a `team_root` and a `gateway_config` to `create_app()` — we will see that pattern in §3.

In [4]:
# The package layout, summarized.
import pkgutil
import arcui as arcui_pkg

modules = sorted(info.name for info in pkgutil.iter_modules(arcui_pkg.__path__))
for m in modules:
    print(f"  arcui.{m}")

  arcui._constants
  arcui.audit
  arcui.auth
  arcui.embedded_agents
  arcui.observe
  arcui.observe_stats
  arcui.query_validators
  arcui.registry
  arcui.routes
  arcui.schemas
  arcui.server
  arcui.team_stream
  arcui.types
  arcui.ws_helpers


Highlights:

- `server.py` — the app factory and uvicorn runner (this notebook)
- `auth.py` — the two-token auth middleware (`AuthConfig`, `AuthMiddleware`, `SessionTracker`)
- `observe.py` — the `Observe` plane: an on-demand arcstore mirror answering trace/stats/agent queries
- `observe_stats.py` — stats/timeseries computation over the mirror (replaces the deleted `RollingAggregator`)
- `registry.py` — `AgentRegistry`, the in-memory online-fleet roster
- `routes/` — every HTTP/WS route the dashboard exposes (covered in §6)
- `audit.py` — `UIAuditLogger` + `UIAuditEvent` taxonomy
- `team_stream.py` — `TeamStreamHub` / `TeamBusObserver` feeding the read-only `/ws/team` stream

## 3. `create_app` — building the Starlette app

`create_app()` is the factory. It returns a fully wired Starlette application that you can hand to **any** ASGI server (uvicorn, hypercorn, daphne) or drive in-process from a test harness.

Calling it with no arguments gives you a self-contained dashboard with auto-generated tokens, no agent fleet, and no trace store:

In [5]:
from arcui import create_app

app = create_app()

print(type(app).__name__)
print("routes registered:", len(app.routes))

Starlette
routes registered: 51


Every piece of plumbing is on `app.state` — that is the contract. Routes do not import the registry, the aggregator, or the trace store directly. They reach into `app.state` so the wiring stays in one place and tests can swap pieces out.

In [6]:
# Inspect the wired-up state. This is the contract every route relies on.
#
# Starlette's `State` object stores attributes in an internal `_state` dict
# and implements `__getattr__`/`__setattr__` but NOT `__dir__` — so
# `dir(app.state)` only shows class-level members, never what's actually
# wired in. `__iter__` over the object *does* yield the real keys; that's
# the supported way to enumerate app.state.
state_keys = [k for k in app.state if not k.startswith("_")]
for k in sorted(state_keys):
    val = getattr(app.state, k)
    print(f"  app.state.{k:24s} -> {type(val).__name__}")

  app.state.agent_info               -> dict
  app.state.agent_registry           -> AgentRegistry
  app.state.audit                    -> UIAuditLogger
  app.state.auth_config              -> AuthConfig
  app.state.circuit_breakers         -> list
  app.state.config_controller        -> NoneType
  app.state.index_html               -> str
  app.state.messaging_service        -> NoneType
  app.state.observe                  -> Observe
  app.state.queue_modules            -> list
  app.state.roster_provider          -> function
  app.state.session_tracker          -> SessionTracker
  app.state.sw_js                    -> str
  app.state.team_post_forwarder      -> NoneType
  app.state.team_root                -> NoneType
  app.state.team_stream              -> TeamStreamHub
  app.state.telemetry_modules        -> list


Several of these matter for bring-up:

- `agent_registry` — tracks online agents (max-cap configurable via `max_agents`)
- `observe` — the `Observe` arcstore mirror; every trace/stats/agent read goes through it
- `audit` — the `UIAuditLogger` that emits `ui.session_start` and friends
- `session_tracker` — at-most-once session-start emission per (token, address)
- `auth_config` — the resolved two-token bundle
- `config_controller` — optional `arcllm.ConfigController` for the Settings page
- `messaging_service` — optional arcteam `MessagingService` powering Team Chat
- `team_stream` — the `TeamStreamHub` feeding `/ws/team`
- `roster_provider` — closure that walks `team_root` to enumerate the fleet

All of these are *optional* to interact with directly. The point of `create_app()` is that the wiring is correct without you touching any of them.

In [7]:
# The factory accepts a handful of keyword arguments that control how
# the dashboard composes with the rest of Arc:
import inspect

from arcui.server import create_app as _factory

sig = inspect.signature(_factory)
for name, param in sig.parameters.items():
    print(f"  {name:20s} default={param.default!r}")

  auth_config          default=None
  config_controller    default=None
  agent_info           default=None
  max_agents           default=100
  team_root            default=None
  gateway_config       default=None
  messaging_service    default=None
  team_post_forwarder  default=None
  team_stream_interval default=1.0
  data_dir             default=None


Each parameter has a clear job:

- `auth_config` — supply your own `AuthConfig`; otherwise tokens are auto-generated.
- `config_controller` — an `arcllm.ConfigController` so the Settings page can mutate runtime config.
- `agent_info` — metadata about *this* agent (when `arcui` is embedded in `arc agent serve`).
- `max_agents` — connection cap for the registry (default `100`).
- `team_root` — path to a `team/` directory; enables the fleet roster and agent-detail endpoints. `None` disables them (empty rosters, 404 for per-agent paths).
- `gateway_config` — when set (and `team_root` is also set), composes an in-process `arcgateway` runtime (executor, session router, web/platform adapters) exposed via `app.state`.
- `messaging_service` — an arcteam `MessagingService`; powers the Team Chat routes. `None` degrades them to empty payloads.
- `team_post_forwarder` — async callable that hands a human group post to arcteam (arcui only forwards, never signs).
- `team_stream_interval` — seconds between `TeamBusObserver` polls feeding `/ws/team`.
- `data_dir` — Arc data directory for the `Observe` mirror (spool + WORM + store); defaults to `arcstore.config.resolve_data_dir()`.

There is no `trace_store=` parameter — trace reads go through `app.state.observe`, which `create_app()` always constructs itself.

In [8]:
# Build the app with explicit knobs to see how it changes.
from unittest.mock import MagicMock

fake_config_ctrl = MagicMock(name="fake_config_controller")
agent_info = {
    "name": "demo-agent",
    "did": "did:arc:local:executor/abc123",
    "model": "anthropic/claude-sonnet-4-6",
    "provider": "anthropic",
}

app2 = create_app(
    config_controller=fake_config_ctrl,
    agent_info=agent_info,
    max_agents=25,
)

print("config_ctrl:    ", app2.state.config_controller)
print("agent_info:     ", app2.state.agent_info)
print("max_agents cap: ", app2.state.agent_registry.max_agents)
print("observe (real Observe mirror, not a mock):", type(app2.state.observe).__name__)

config_ctrl:     <MagicMock name='fake_config_controller' id='4481906496'>
agent_info:      {'name': 'demo-agent', 'did': 'did:arc:local:executor/abc123', 'model': 'anthropic/claude-sonnet-4-6', 'provider': 'anthropic'}
max_agents cap:  25
observe (real Observe mirror, not a mock): Observe


`config_controller` is a `MagicMock` stand-in here because `create_app()` is a *wiring* function — it stores the reference, it does not call into it. `observe`, by contrast, is always the real `Observe` mirror; `create_app()` builds it unconditionally from `data_dir` (or the default arcstore data dir). We exercise it via `TestClient` in §5.

## 4. The two-token auth model

Most observability dashboards ship with a single shared password. That fails one predictable way: the read-only viewer who sees a chart can also push admin commands.

`arcui` solves it with two tokens:

In [9]:
from arcui.auth import AuthConfig

# Auto-generate. Each token is 32 random bytes hex-encoded (64 chars).
cfg = AuthConfig()

print("viewer_token len  :", len(cfg.viewer_token))
print("operator_token len:", len(cfg.operator_token))
print("distinct           :", cfg.viewer_token != cfg.operator_token)

viewer_token len  : 64
operator_token len: 64
distinct           : True


Each token maps to one role. The mapping is enforced by `AuthMiddleware`, which sits in front of every `/api/*` route:

| Token | Role | Allowed |
|---|---|---|
| viewer | read-only | dashboard, traces, stats, `arc ui tail` |
| operator | read + control | viewer + approve pairings, admin actions |

Validation is `hmac.compare_digest` so a network-timing attack cannot enumerate the token by length-of-prefix. There used to be a third "agent" token gating a live agent-push WebSocket (`/api/agent/connect`); that channel was removed in SPEC-026's push-pipeline teardown, so the auth model shrank to two roles.

In [10]:
# Supply your own tokens (e.g. from a secrets manager) instead of
# auto-generating. The constructor accepts a plain dict so an Ansible
# playbook or systemd unit can hand over secrets without importing
# anything Arc-specific.
supplied = AuthConfig(
    {
        "viewer_token": "viewer-demo-token",
        "operator_token": "operator-demo-token",
    }
)

print("viewer   ->", supplied.validate_token("viewer-demo-token"))
print("operator ->", supplied.validate_token("operator-demo-token"))
print("unknown  ->", supplied.validate_token("not-a-real-token"))

viewer   -> viewer
operator -> operator
unknown  -> None


Pass the resolved `AuthConfig` to `create_app(auth_config=...)`. From that point on, the middleware enforces the policy on every request.

In [11]:
# Build an app with our known tokens and inspect the middleware layer.
from arcui.auth import AuthMiddleware

app_with_auth = create_app(auth_config=supplied)

middleware_classes = [m.cls.__name__ for m in app_with_auth.user_middleware]
print("middleware stack:", middleware_classes)
assert "AuthMiddleware" in middleware_classes

middleware stack: ['AuthMiddleware']


**No token file on disk.** Tokens are never persisted to a file. `arc ui start` prints both tokens to the console, masked unless `--show-tokens` is passed. On a loopback bind (`127.0.0.1`/`localhost`), it also auto-opens the browser with the viewer token embedded in the URL hash fragment and calls `session_tracker.mark_bootstrap_issued(viewer_token)` so `AuthMiddleware` records that first request as `auth_method="browser_bootstrap"` rather than `"manual_token"`. Only the *viewer* token is ever auto-delivered this way — the operator token is for a human to copy-paste deliberately.

## 5. Bringing up the server — `serve()` and `TestClient`

`serve()` is the one-liner. It:

1. Calls `create_app()` with your kwargs.
2. Optionally calls `attach_llm(app, llm)` if you passed an `arcllm` model — this walks the module stack for circuit-breaker/telemetry/queue introspection; it does not warm anything from a trace store (there isn't one).
3. Hands the app to `uvicorn.run(host, port)`.

Step 3 is a *blocking* call. In a notebook we never want to block — so we drive the app **in-process** via `starlette.testclient.TestClient`, which goes through the full ASGI stack (including the lifespan, which starts the `Observe` mirror) but never opens a real socket.

In [12]:
import inspect

print(inspect.signature(serve))
print()
print(inspect.getdoc(serve))

(llm: 'Any' = None, *, host: 'str' = '127.0.0.1', port: 'int' = 8420, config_controller: 'Any | None' = None, auth_config: 'AuthConfig | None' = None) -> 'None'

One-liner to start ArcUI dashboard.

Usage::

    from arcui import serve
    serve(llm=model)

Args:
    llm: Optional LLMProvider to attach immediately.
    host: Bind address (default localhost).
    port: Port number (default 8420).
    config_controller: ArcLLM ConfigController for config management.
    auth_config: Auth configuration. Auto-generated if None.


Same kwargs as `create_app()`, plus `host`, `port`, and an `llm`. The defaults are `127.0.0.1:8420` — local-by-default, never `0.0.0.0`. We will come back to that choice in §8.

In [13]:
# Drive the app in-process with TestClient. No socket is bound.
from starlette.testclient import TestClient

auth = AuthConfig(
    {
        "viewer_token": "viewer-tok",
        "operator_token": "operator-tok",
    }
)
tc_app = create_app(auth_config=auth)
client = TestClient(tc_app)

# Liveness probe — exempt from auth.
resp = client.get("/api/health")
print("GET /api/health:", resp.status_code, resp.json())

GET /api/health: 200 {'status': 'ok'}


/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/ipykernel_22272/3177168326.py:2: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient


Health is exempt by design — Kubernetes liveness probes must not need credentials. Every other `/api/*` route does require a token.

In [14]:
# Without a token: 401.
resp = client.get("/api/info")
print("no token:    ", resp.status_code, resp.json())

# Wrong token: 401.
resp = client.get("/api/info", headers={"Authorization": "Bearer not-real"})
print("bad token:   ", resp.status_code, resp.json())

# Viewer token: 200.
resp = client.get("/api/info", headers={"Authorization": "Bearer viewer-tok"})
print("viewer ok:   ", resp.status_code, resp.json())

auth.missing_token path=/api/info


auth.invalid_token path=/api/info


no token:     401 {'error': 'Missing token'}
bad token:    401 {'error': 'Invalid token'}
viewer ok:    200 {}


Now the operator-only rule (ASI-03 — privilege abuse). `AuthMiddleware` only decides *whether* a token is valid and which role it maps to (`request.state.role`); it's up to each route to check the role for anything beyond plain read access. `PATCH /api/config` is one such route — it requires `role == "operator"`. Using a viewer token there is a 403, not a 401: the credential is valid, the role just isn't allowed to mutate config.

In [15]:
# viewer token on an operator-only route -> 403 (valid credential, wrong role).
# No config_controller is wired on this app, but the role check runs first.
resp = client.patch(
    "/api/config",
    headers={"Authorization": "Bearer viewer-tok"},
    json={"some": "update"},
)
print("viewer on operator-only route:", resp.status_code, resp.json())

resp = client.patch(
    "/api/config",
    headers={"Authorization": "Bearer operator-tok"},
    json={"some": "update"},
)
print("operator on operator-only route:", resp.status_code, resp.json())

viewer on operator-only route: 403 {'error': 'Operator role required'}
operator on operator-only route: 404 {'error': 'No config controller configured'}


Now exercise the `agent_info` surface — the `/api/info` endpoint reflects whatever metadata you passed at construction. This is what the dashboard reads to render the agent's name and DID in the title bar:

In [16]:
info_app = create_app(
    auth_config=auth,
    agent_info={
        "name": "demo-agent",
        "did": "did:arc:local:executor/abc123",
        "model": "anthropic/claude-sonnet-4-6",
    },
)
info_client = TestClient(info_app)

resp = info_client.get("/api/info", headers={"Authorization": "Bearer viewer-tok"})
print(resp.status_code)
for k, v in resp.json().items():
    print(f"  {k:8s} {v}")

200
  name     demo-agent
  did      did:arc:local:executor/abc123
  model    anthropic/claude-sonnet-4-6


**The subprocess pattern.** When you genuinely need to verify a real port-bound bringup (smoke-test, packaging release gate), run `serve()` in a subprocess with a hard timeout. The cell below documents the shape; we leave it commented so the notebook never actually binds a port:

In [17]:
# Documented pattern — DO NOT uncomment in this notebook.
#
# import subprocess, sys, time
#
# proc = subprocess.Popen(
#     [sys.executable, "-c",
#      "from arcui import serve; serve(host='127.0.0.1', port=18420)"],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
# )
# try:
#     time.sleep(1.0)            # wait for uvicorn boot
#     # ... hit http://127.0.0.1:18420/api/health with httpx ...
# finally:
#     proc.terminate()
#     proc.wait(timeout=5)
print("subprocess pattern documented — TestClient is preferred for notebook tests.")

subprocess pattern documented — TestClient is preferred for notebook tests.


`TestClient` is preferred because it covers everything except `uvicorn.run()` itself — every route, every middleware, every lifespan startup hook fires. The subprocess is only useful when the *exact* `uvicorn` binding behavior is what you are testing.

## 6. Routing structure — what the dashboard exposes

`create_app()` registers routes in a fixed order. We can enumerate them off the live app:

In [18]:
# Pull the route table off the live app.
from starlette.routing import Mount, Route, WebSocketRoute

summary = []
for r in app.routes:
    if isinstance(r, Route):
        methods = ",".join(sorted(r.methods or set())) if r.methods else "-"
        summary.append(("HTTP", r.path, methods))
    elif isinstance(r, WebSocketRoute):
        summary.append(("WS", r.path, "-"))
    elif isinstance(r, Mount):
        summary.append(("MOUNT", r.path, "-"))

print(f"Total routes: {len(summary)}\n")
for kind, path, methods in summary[:25]:
    print(f"  {kind:6s} {path:48s} {methods}")
print("  ...")

Total routes: 51

  HTTP   /                                                GET,HEAD
  HTTP   /sw.js                                           GET,HEAD
  HTTP   /api/health                                      GET,HEAD
  HTTP   /api/info                                        GET,HEAD
  HTTP   /api/traces                                      GET,HEAD
  HTTP   /api/traces/{trace_id}                           GET,HEAD
  HTTP   /api/config                                      GET,HEAD
  HTTP   /api/config                                      PATCH
  HTTP   /api/arcllm-config                               GET,HEAD
  HTTP   /api/arcllm-config                               PATCH
  HTTP   /api/stats                                       GET,HEAD
  HTTP   /api/stats/timeseries                            GET,HEAD
  HTTP   /api/circuit-breakers                            GET,HEAD
  HTTP   /api/budget                                      GET,HEAD
  HTTP   /api/performance                         

Grouped by purpose, the surface is small and predictable:

| Group | Examples | Source module |
|---|---|---|
| Liveness / metadata | `/api/health`, `/api/info` | `server.py` |
| Dashboard SPA | `/`, `/sw.js`, `/assets/*` | `server.py` |
| Trace queries | `/api/traces`, `/api/traces/{id}` | `routes.traces` |
| Stats & cost | `/api/stats`, `/api/stats/timeseries`, `/api/cost-efficiency` | `routes.stats`, `routes.cost_efficiency` |
| Config | `/api/config`, `/api/arcllm-config` | `routes.config`, `routes.arcllm_config` |
| Agent fleet | `/api/agents`, `/api/agents/{id}`, team pages | `routes.agents`, `routes.agent_detail`, `routes.team_pages` |
| Knowledge | agent-detail knowledge sub-routes | `routes.knowledge` |
| Export | `/api/export/*` | `routes.export` |
| Runs / observe | run + spawn-tree queries over the mirror | `routes.observe_run` |
| Team chat | `/api/team/*` chat endpoints | `routes.team_chat` |
| WebSockets | `/ws/chat/{agent_id}` (interactive chat, bidirectional), `/ws/team` (read-only team-flow stream) | `routes.chat_ws`, `routes.team_ws` |

In [19]:
# Quick sanity-check that the well-known REST + WS endpoints are all wired.
expected = [
    "/",
    "/api/health",
    "/api/info",
    "/api/traces",
    "/api/agents",
    "/ws/chat/{agent_id}",
    "/ws/team",
]
paths = {r.path for r in app.routes if hasattr(r, "path")}
for p in expected:
    print(f"  {p:24s} {'OK' if p in paths else 'MISSING'}")

  /                        OK
  /api/health              OK
  /api/info                OK
  /api/traces              OK
  /api/agents              OK
  /ws/chat/{agent_id}      OK
  /ws/team                 OK


`/ws/chat/{agent_id}` is the interactive chat WebSocket — bidirectional, one per agent. `/ws/team` is read-only, fed by `TeamBusObserver` polling the arcteam bus. Both require a valid viewer/operator token like any other route; neither is a push channel for arbitrary agent telemetry — that channel (`/api/agent/connect`) was removed in the SPEC-026 teardown.

In [20]:
# Example: list traces from an empty Observe mirror (no data_dir with real
# arcstore data in this environment). Returns an empty page, not an error.
resp = client.get("/api/traces", headers={"Authorization": "Bearer viewer-tok"})
print(resp.status_code)
print(resp.json())

200
{'traces': [{'trace_id': '20cb972bb65f21088fdd81decc9894ed', 'timestamp': '2026-07-09T12:21:09.082016+00:00', 'model': 'gemini-2.5-flash', 'provider': 'google', 'agent': 'did:arc:unknown', 'agent_label': None, 'status': 'success', 'cost_usd': 1.9050000000000002e-05, 'duration_ms': 610.7, 'input_tokens': 51, 'output_tokens': 19, 'total_tokens': 70, 'request_id': 'a275f099-0189-4e2d-87b5-0a1f47836370', 'request': {'messages': [{'role': 'system', 'content': 'You are concise. Answer in one sentence.'}, {'role': 'user', 'content': 'Summarize what an agent loop does.'}], 'tools': [{'name': 'noop', 'description': 'A no-op tool — never called by the model.', 'parameters': {'type': 'object', 'properties': {}, 'required': []}}]}, 'response': {'content': 'An agent loop continuously observes the environment, decides on an action, and then executes that action.', 'tool_calls': [], 'stop_reason': 'end_turn'}, 'messages': [{'role': 'system', 'content': 'You are concise. Answer in one sentence.'},

Point `data_dir` at a directory with real arcstore spool/WORM data and the same endpoint returns real records — `Observe.start()` backfills from it during the lifespan. We will exercise that path properly in `02-live-telemetry-attach.ipynb`.

In [21]:
# The dashboard SPA itself. Index serves the cached HTML; static assets
# are mounted under /assets so the browser fetches JS/CSS without
# touching the auth middleware.
resp = client.get("/")
ct = resp.headers.get("content-type", "")
cc = resp.headers.get("cache-control", "")
print(f"GET / status={resp.status_code} content-type={ct!r} cache-control={cc!r}")

GET / status=200 content-type='text/html; charset=utf-8' cache-control='no-store, no-cache, must-revalidate'


`Cache-Control: no-store` on the SPA shell means a stale tab cannot survive an `arc ui start` restart with old asset references — important because every restart issues fresh tokens, and a stale tab serving cached JS would call dead endpoints.

## 7. Compliance hooks — what auditors care about

The README has the canonical mapping, but two things are worth surfacing here because they show up in the wiring you just built:

**NIST 800-53:**

| Control | Where it lives in `create_app()` |
|---|---|
| AU-2 (audit events) | `app.state.audit` (`UIAuditLogger`) — emits `ui.session_start`, `ui.agent_autoconnect`, `auth.failure`, `auth.success`, `auth.rejected` (`UIAuditEvent` taxonomy). |
| AU-9 (protection of audit information) | The dashboard reads through `app.state.observe`; there is no write path from the UI back into the durable spool/WORM store. |
| AU-11 (retention) | The durable record lives in arcstore's spool + WORM store (outside `arcui`); `Observe` is a read-only mirror over it, not the system of record. |
| SI-4 (continuous monitoring) | `/ws/team` streams team-flow events as `TeamBusObserver` polls the arcteam bus; REST reads are on-demand against the mirror rather than a live push feed. |
| SC-13 (cryptographic role separation) | `AuthConfig` — two tokens, one role each, `hmac.compare_digest` validation. |

**OWASP Agentic 2026:**

| Risk | Mitigation in `arcui` |
|---|---|
| ASI-03 (Identity & Privilege Abuse) | Viewer tokens are read-only; operator-only routes (e.g. `PATCH /api/config`) check `request.state.role` explicitly and 403 a valid-but-wrong-role token. |
| ASI-09 (Trust Exploitation) | Every agent action surfaces with `agent_id` + `did` attribution in the durable record; humans can see impersonation after the fact. |
| ASI-10 (Rogue Agents) | The dashboard *is* the behavioral monitoring surface over the durable record; anomalies show up on the next read. |

In [22]:
# Confirm the audit logger is wired and emitting structured events.
import logging

logger = app.state.audit
print("logger type:", type(logger).__name__)

# UIAuditLogger writes structured records to the configured logger.
# Plumbing-level smoke: the logger object exists, has the expected
# audit_event method.
assert hasattr(logger, "audit_event"), "audit logger must expose audit_event()"
print("audit_event method present: OK")

logger type: UIAuditLogger
audit_event method present: OK


The `audit_event()` method is what the auth middleware calls when it hands out a fresh `session_id` to a new (token, remote_addr) pair. SR-3 / SPEC-025 §FR-7 mandate five fields per event — `session_id`, `uid`, `username`, `remote_addr`, `auth_method` — encoded as a Pydantic model so dropping a field is a type error, not a silent audit gap.

In [23]:
# The session tracker is the at-most-once gate.
from arcui.auth import SessionTracker

tracker = app.state.session_tracker
print("tracker type:", type(tracker).__name__)
print("is SessionTracker:", isinstance(tracker, SessionTracker))

tracker type: SessionTracker
is SessionTracker: True


Two interesting properties of `SessionTracker`:

1. **Token hashing.** Tokens are SHA-256 hashed before storage so a memory dump or audit log of session ids cannot be reversed to the bearer token (SR-2).
2. **LRU bounding.** Both internal stores are bounded LRUs; on overflow the oldest entry is evicted, which re-emits `ui.session_start` for a long-idle client when it returns. That re-emission is the *correct* audit behavior, not a bug — it preserves the chain across long quiet periods.

In [24]:
# The bounds are env-configurable for federal tuning.
import os

for var in ("ARCUI_MAX_SESSIONS", "ARCUI_MAX_BOOTSTRAP_MARKERS"):
    print(f"  {var:32s} -> {os.environ.get(var, '(unset, default)')}")

  ARCUI_MAX_SESSIONS               -> (unset, default)
  ARCUI_MAX_BOOTSTRAP_MARKERS      -> (unset, default)


The defaults — 10 000 sessions, 1 000 bootstrap markers — cap memory at well under 5 MB and are safe for a long-running federal deployment without explicit tuning.

## 8. Operational notes — TLS, binding, reverse proxy, port choice

`serve()` is *intentionally* opinionated:

- **Bind defaults to `127.0.0.1`.** Not `0.0.0.0`. The dashboard is local-by-default. To bind all interfaces you must pass `host='0.0.0.0'` explicitly — and you should put it behind mTLS or a reverse proxy when you do.
- **No TLS in `serve()` itself.** `serve()` runs plain HTTP. For TLS, terminate at a reverse proxy (Caddy, nginx, Traefik) or hand the app to a uvicorn invocation with `ssl_keyfile=` / `ssl_certfile=`.
- **Port `8420` by default.** Why 8420: outside common reserved ranges, easy to remember, and matches the rest of the Arc tooling.
- **Single-process.** `uvicorn.run(app, ...)` runs one worker. To horizontally scale, front the dashboard with a process manager that points many workers at a shared arcstore data dir.

In [25]:
# Verify the defaults match the README and the docstring.
import inspect

params = inspect.signature(serve).parameters
print("host default:", params["host"].default)
print("port default:", params["port"].default)

# Local-by-default invariant.
assert params["host"].default == "127.0.0.1", "federal-first: bind localhost by default"
assert params["port"].default == 8420, "port choice changed — update README"

host default: 127.0.0.1
port default: 8420


Keeping `serve()` opinionated about binding is a federal-mode invariant. It is the difference between a developer accidentally exposing live agent telemetry on a lab subnet vs. forcing them to write `--host 0.0.0.0` deliberately and notice the choice.

In [26]:
# Reverse-proxy front: the app is plain ASGI; pass it to uvicorn yourself
# when you need control over workers, ssl, or proxy headers.
#
# Example invocation (do NOT execute in the notebook):
#
#   uvicorn arcui.server:create_app --factory \
#       --host 0.0.0.0 --port 8420 \
#       --workers 1 \
#       --proxy-headers --forwarded-allow-ips '127.0.0.1' \
#       --log-level info
#
# Then in nginx / Caddy:
#
#   reverse_proxy 127.0.0.1:8420 {
#       header_up X-Forwarded-Proto {scheme}
#       header_up Host             {host}
#   }
print("see comment for the canonical reverse-proxy invocation")

see comment for the canonical reverse-proxy invocation


**Why `--proxy-headers` matters.** `arcui` reads `request.client.host` to attribute audit events. Without `--proxy-headers` the audit log shows the proxy's IP for every viewer. With it, uvicorn unwraps `X-Forwarded-For` and you see the real client.

**Why `--workers 1`.** `agent_registry`, `session_tracker`, and `team_stream` are all *in-process* state. Two uvicorn workers would fragment the online-fleet roster and session tracking across processes, and each worker would run its own `TeamBusObserver`. To scale horizontally, run multiple `arcui` instances behind a load balancer with sticky sessions on the WebSocket endpoints and a shared arcstore data dir — `Observe` reads are stateless per request, so that part scales cleanly; the in-process bits don't.

In [27]:
# Sanity-check that the package re-exports stay tight.
from arcui import __all__ as exported

expected = {"__version__", "create_app", "serve", "attach_llm"}
actual = set(exported)
missing = expected - actual
extra = actual - expected
print("missing:", missing or "none")
print("extra:  ", extra or "none")
assert not missing, f"public surface lost a name: {missing}"
assert not extra, f"public surface grew unexpectedly: {extra}"

missing: none
extra:   none


If this assertion ever fires, the public surface has drifted and either `__init__.py` or this notebook needs updating. The point of pinning the surface in a smoke test is that drift cannot land silently.

In [28]:
# Tear down. TestClient holds an internal task group; closing it is hygienic
# even though the next garbage-collection pass would do it for us.
client.close()
info_client.close()
print("clients closed; notebook is ready for the next walkthrough.")

clients closed; notebook is ready for the next walkthrough.


## 9. Summary

We brought the dashboard up three different ways and never opened a real port:

- **`create_app()`** is the Starlette factory. Notable optional kwargs: `auth_config`, `agent_info`, `team_root`, `gateway_config`, `messaging_service`, `data_dir`. There is no `trace_store=` — `app.state.observe` (the arcstore mirror) is always constructed internally. Everything is wired onto `app.state` so routes never reach for module-level singletons.
- **`serve()`** is the one-liner that bundles `create_app()` + optional `attach_llm()` + `uvicorn.run()`. Local-by-default (`127.0.0.1:8420`), no TLS — terminate TLS at a reverse proxy.
- **Two tokens, two roles.** `viewer` reads, `operator` reads + controls (route-level checks like `PATCH /api/config` enforce the operator-only ones). `hmac.compare_digest` validation. The old third "agent" token and its push channel (`/api/agent/connect`) are gone (SPEC-026).
- **`TestClient` is the right notebook tool.** It exercises the full ASGI stack — middleware, lifespan (including `Observe.start()`), routes — without binding a socket. Subprocess + timeout is documented but reserved for packaging-level smoke tests.
- **Routing surface is small and stable.** `/api/health`, `/api/info`, traces, stats, agents, config, export, team chat, plus two WebSocket endpoints: `/ws/chat/{agent_id}` (bidirectional) and `/ws/team` (read-only).
- **Compliance hooks live in the wiring.** `UIAuditLogger` (`UIAuditEvent` taxonomy), `SessionTracker` with hashed tokens and bounded LRUs, mandatory five-field session-start events. NIST AU-2/9/11, SI-4, SC-13. OWASP Agentic ASI-03 / ASI-09 / ASI-10.
- **Operational invariants.** Bind localhost by default, single uvicorn worker (in-process registry/tracker/team-stream state), terminate TLS upstream, rely on `--proxy-headers` for accurate audit attribution.

**Public API exercised in this notebook:** `arcui.__version__`, `arcui.create_app`, `arcui.serve`, `arcui.attach_llm` (referenced; full coverage in 02), `arcui.auth.AuthConfig`, `arcui.auth.AuthMiddleware`, `arcui.auth.SessionTracker`.

**Next: see `02-live-telemetry-attach.ipynb`** — wires an `arcllm` model into the running app via `attach_llm`, walks the `/ws/chat` protocol, and exercises trace + stat queries against a populated `Observe` mirror.